In [ ]:
# =============================================================================
# CELL 1: SETUP AND CONFIGURATION
# =============================================================================
# This cell initialises the environment, defines all dynamic variables,
# and sets the configurations required for the pipeline orchestration.
# Users must ensure these variables accurately reflect their specific dataset.

import os
from src.config_manager import (
    DataConfig,
    ColumnConfig,
    LLMConfig,
    PipelineRulesConfig,
    TaxonomyConfig,
    RedactionConfig
)

# 1. API Authentication
# While you can paste your key directly here for rapid prototyping,
# it is recommended to use environment variables for security.
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Export it before running the notebook.")
os.environ["OPENAI_API_KEY"] = api_key

# 2. File Paths
# Define the relative or absolute paths to your raw logs and taxonomy dictionaries.
INPUT_DATA_PATH = "data/raw/maintenance_logs.csv"
TAXONOMY_PATH = "data/taxonomy/rds_pp_components.csv"
OUTPUT_DATA_PATH = "data/processed/structured_logs_labelled.csv"

# 3. Dynamic Column Mapping
# Customise these strings to match the exact column headers in your input CSV.
column_config = ColumnConfig(
    wo_id_col="WO_ID",
    system_code_col="System_Code",
    system_name_col="System_Name",
    description_col="Descriptions",
    
    # Optional legacy categorical fields
    maintenance_type_col="Maintenance_Type",
    action_type_col="Action_Type",
    
    # Turbine technical specifications (used for context injection)
    manufacturer_col="Manufacturer",
    turbine_model_col="Turbine_Model",
    drive_type_col="Drive_Type",
    generator_type_col="Generator_Type",
    pitch_type_col="Pitch_Type",
    yaw_brake_type_col="Yaw_Brake_Type"
)

# 4. Large Language Model Configuration
# Adjust the model version, reasoning parameters, and API pricing limits.
llm_config = LLMConfig(
    model="gpt-5.4",
    reasoning_effort="medium",
    phase3_reasoning_effort="high",
    max_concurrent_requests=10, # Reduce this if you encounter rate limits
    phase3_max_concurrent_requests=5,
    timeout_seconds=60.0,
    max_retries=3,
    initial_backoff_seconds=1.0,
    max_backoff_seconds=20.0,
    backoff_multiplier=2.0,
    jitter_seconds=0.5,
    input_cost_per_million=2.50,
    output_cost_per_million=15.00
)

# 5. Pipeline Rules and Validation Thresholds
# Control the deterministic behaviour, cost-saving skip rules, and QA thresholds.
rules_config = PipelineRulesConfig(
    required_confidence_level="High",
    allow_human_review_flags=False,
    skip_preventive_maintenance=True,
    skip_missing_descriptions=True,
    garbage_data_label="Not Enough Information",
    healthy_turbine_label="Not Applicable",
    non_failure_maintenance_types=["Preventive", "Retrofit"],
    max_row_retries=2
)

# 6. Taxonomy Branch Selection (Optional)
# Provide facility/system prefixes to narrow Phase 1 classification.
# Leave as None to use the full taxonomy.
taxonomy_config = TaxonomyConfig(
    facility_prefix=None,
    system_prefix=None,
    # Required for isolated pitch-only and restricted non-pitch routing.
    # Adjust these prefixes to match your RDS-PP taxonomy variant.
    pitch_system_prefixes=["MDA"]
)

# 7. Redaction/Anonymisation (Optional)
# Add regex patterns or a custom callable to redact sensitive data.
redaction_config = RedactionConfig(
    enabled=False,
    patterns=[],
    replacement="[REDACTED]",
    custom_redactor=None
)

# 8. Master Data Configuration
# Consolidate the configurations to pass down to the orchestrator engine.
master_config = DataConfig(
    columns=column_config,
    llm=llm_config,
    rules=rules_config,
    taxonomy=taxonomy_config,
    redaction=redaction_config,
    default_input_dir=os.path.dirname(INPUT_DATA_PATH),
    default_output_dir=os.path.dirname(OUTPUT_DATA_PATH)
)

print("Configuration initialised successfully. The pipeline is ready to be instantiated.")

In [ ]:
# =============================================================================
# CELL 2: CORE SYSTEM INITIALISATION
# =============================================================================
# Instantiate the backend modules required for the labelling pipeline.

from src.data_manager import DataManager
from src.taxonomy_handler import TaxonomyHandler
from src.llm_clients.openai_client import AsyncOpenAIClient
from src.pipeline_orchestrator import PipelineOrchestrator

print("Initialising core system components...")

# 1. Data I/O Manager
data_manager = DataManager(config=master_config)

# 2. Taxonomy Manager
taxonomy_handler = TaxonomyHandler(taxonomy_path=TAXONOMY_PATH)

# 3. LLM API Client
api_client = AsyncOpenAIClient(config=master_config.llm)

# 4. Pipeline Orchestrator
# The chunk_size determines how many rows are processed before saving to disk.
orchestrator = PipelineOrchestrator(
    config=master_config,
    data_manager=data_manager,
    api_client=api_client,
    taxonomy_handler=taxonomy_handler,
    chunk_size=50
)
orchestrator.initialize_performance_logger(master_config.default_output_dir)

print("System components initialised successfully.")

In [ ]:
# =============================================================================
# CELL 3: DATA INGESTION AND VALIDATION
# =============================================================================
# Load the raw operational dataset and verify that the defined column names
# exist within the file to prevent downstream pipeline errors.

import pandas as pd

print(f"Loading operational data from: {INPUT_DATA_PATH}")

# Define the absolute minimum columns required to proceed
required_columns = [
    master_config.columns.wo_id_col,
    master_config.columns.description_col
]

try:
    # Load the dataset using the robust DataManager
    raw_logs_df = data_manager.load_dataset(
        file_path=INPUT_DATA_PATH,
        required_columns=required_columns
    )
    
    print(f"Dataset loaded successfully with {len(raw_logs_df)} records.")
    print("Displaying the first 5 records:")
    
    # Display the dataframe head for visual confirmation
    display(raw_logs_df.head())
    
except Exception as e:
    print(f"Data Ingestion Error: {e}")
    print("Please correct your dataset or adjust the ColumnConfig in Cell 1.")

In [ ]:
# =============================================================================
# CELL 4: TAXONOMY VERIFICATION
# =============================================================================
# Load the hierarchical taxonomy and print a sample branch to ensure
# the tree is constructed properly before injecting it into the LLM prompts.

print(f"Loading taxonomy from: {TAXONOMY_PATH}")

try:
    # Parse the CSV and build the nested dictionary tree
    taxonomy_handler.load_and_parse_taxonomy()
    
    # Retrieve a sample branch to verify functionality, such as the Pitch System 'MDA'.
    # Adjust the prefixes below if your taxonomy uses a different structure.
    sample_facility = "=G001"
    sample_system = "MDA"
    
    print(f"\nRetrieving sample taxonomy branch for {sample_facility} {sample_system}...")
    sample_branch = taxonomy_handler.get_system_branch(sample_facility, sample_system)
    
    if sample_branch:
        formatted_sample = taxonomy_handler.format_branch_for_prompt(sample_branch)
        print("\nTaxonomy Tree Sample:")
        print("-" * 50)
        print(formatted_sample)
        print("-" * 50)
    else:
        print(f"Warning: Branch {sample_facility} {sample_system} not found. Please verify your taxonomy structure.")
        
except Exception as e:
    print(f"Taxonomy Parsing Error: {e}")

In [ ]:
# =============================================================================
# CELL 5: SYSTEM CODE CORRECTIONS AND AUDIT (PHASES 1 & 2)
# =============================================================================
# Execute the system classification tasks. The pipeline will automatically chunk
# the data, process it asynchronously, and save progress to the checkpoint files.

import os

print("Initiating Phase 1: System Code Corrections...")
cp_p1_path = os.path.join(master_config.default_output_dir, "checkpoint_phase1.csv")

# Execute Phase 1 asynchronously
df_phase_1 = await orchestrator.run_phase_1_system_code_corrections(
    df=raw_logs_df, 
    checkpoint_path=cp_p1_path
)

print("\nInitiating Phase 2: System Code Audit...")
cp_p2_path = os.path.join(master_config.default_output_dir, "checkpoint_phase2.csv")

# Execute Phase 2 asynchronously using the output from Phase 1
df_phase_2 = await orchestrator.run_phase_2_system_code_audit(
    df=df_phase_1, 
    checkpoint_path=cp_p2_path
)

print("\nInitiating Phase 2b: Audit Relabelling...")
cp_p2_relabel_path = os.path.join(master_config.default_output_dir, "checkpoint_phase2_relabel.csv")

# Relabel only the rows flagged by the audit using the full taxonomy
df_phase_2 = await orchestrator.run_phase_2_relabel_failed_audits(
    df=df_phase_2,
    checkpoint_path=cp_p2_relabel_path
)

# Apply threshold-gated system-code merges before Phase 3 so semantic extraction
# uses the corrected active system code where available.
df_phase_2 = orchestrator.apply_system_code_merges(df_phase_2)

print("\nPhases 1, 2, and 2b complete. Displaying a sample of the audited system codes:")
display(df_phase_2[[
    master_config.columns.wo_id_col, 
    master_config.columns.system_code_col, 
    "P1_Reasoning", 
    "P2_Requires_Relabel",
    "P2_Relabel_Suggested_Code",
    "Merged_System_Code",
    "System_Code_Corrected_Flag"
]].head())

In [ ]:
# =============================================================================
# CELL 6: SEMANTIC BATCH EXTRACTION (PHASE 3)
# =============================================================================
# Extract empirical Failure Modes and Effects Analysis (FMEA) dictionaries 
# directly from the text descriptions of the audited logs.

import os
import json

print("Initiating Phase 3: Semantic Batch Extraction...")

# Define the path to cache the extracted dictionaries, saving API costs on reruns.
dict_path = os.path.join(master_config.default_output_dir, "fmea_dictionaries.json")

# Execute Phase 3 asynchronously
fmea_dictionaries = await orchestrator.run_phase_3_batch_extraction(
    df=df_phase_2,
    dict_path=dict_path
)

print("\nPhase 3 complete. Review the empirically generated FMEA dictionaries below:")

# Pretty-print the dictionary to allow the researcher to verify the logic
print(json.dumps(fmea_dictionaries, indent=4))

# Optional manual override:
# If the generated dictionary contains errors, you can manually modify the 
# 'fmea_dictionaries' variable in a new cell before running Cell 7.

In [ ]:
# =============================================================================
# CELL 7: GRANULAR LABELLING (PHASE 4)
# =============================================================================
# Apply log-by-log labelling using the dictionaries generated in Phase 3.
# Deterministic skip rules and dynamic topology injection are applied here.

print("Initiating Phase 4: Granular Operational and FMEA Labelling...")
cp_p4_path = os.path.join(master_config.default_output_dir, "checkpoint_phase4.csv")

# Execute Phase 4 asynchronously
final_labelled_df = await orchestrator.run_phase_4_granular_labelling(
    df=df_phase_2, 
    fmea_dictionaries=fmea_dictionaries,
    checkpoint_path=cp_p4_path
)

# Apply threshold-gated granular merges before QA and export.
final_labelled_df = orchestrator.apply_granular_merges(final_labelled_df)

print("\nPhase 4 complete. The dataset is fully structured.")
print("Displaying a sample of the extracted failure mechanisms and review flags:")

# Define the columns to display for the final check
display_cols = [
    master_config.columns.wo_id_col,
    "P4_Maintenance_Type",
    "P4_Failure_Mode",
    "P4_Cause",
    "P4_Confidence",
    "P4_Review_Flag"
]

display(final_labelled_df[display_cols].head(10))

In [ ]:
# =============================================================================
# CELL 8: QUALITY ASSURANCE, ANALYTICS, AND FINAL EXPORT
# =============================================================================
# Apply post-processing QA filters, review pipeline execution metrics,
# visualise the pass/fail distribution, and confirm the final dataset export.

import os
import pandas as pd
import matplotlib.pyplot as plt

print("Applying Post-Processing QA Filters...\n")

# 1. Apply Quality Filters based on PipelineRulesConfig
final_qa_df = orchestrator.apply_quality_filters(final_labelled_df)

print("\nGenerating post-execution analytics...\n")

# 2. Cost and Performance Reporting
perf_log_path = os.path.join(master_config.default_output_dir, "performance_log.csv")

if os.path.exists(perf_log_path):
    try:
        perf_df = pd.read_csv(perf_log_path)
        
        total_input = perf_df["Input_Tokens"].sum()
        total_output = perf_df["Output_Tokens"].sum()
        total_cost = perf_df["Cost_USD"].sum()
        total_runtime = perf_df["Runtime_Seconds"].sum() / 60  # Convert to minutes
        
        print("-" * 50)
        print("PIPELINE PERFORMANCE SUMMARY")
        print("-" * 50)
        print(f"Total Input Tokens:  {total_input:,}")
        print(f"Total Output Tokens: {total_output:,}")
        print(f"Total Runtime:       {total_runtime:.2f} minutes")
        print(f"Total API Cost:      ${total_cost:.4f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"Error reading performance log: {e}\n")
else:
    print("Performance log not found. Cost metrics are unavailable.\n")

# 3. Visualise QA Status Distribution
if "QA_Status" in final_qa_df.columns:
    qa_counts = final_qa_df["QA_Status"].value_counts(dropna=False)
    
    plt.figure(figsize=(10, 5))
    qa_counts.plot(
        kind="barh", 
        color=["#4CAF50" if "Pass" in str(idx) else "#F44336" for idx in qa_counts.index],
        edgecolor="black"
    )
    plt.title("Post-Processing QA Status Distribution", fontsize=14)
    plt.xlabel("Number of Logs", fontsize=12)
    plt.ylabel("QA Status", fontsize=12)
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("QA Status is not available for plotting.")

# 4. Final Data Export
try:
    os.makedirs(os.path.dirname(OUTPUT_DATA_PATH), exist_ok=True)
    final_qa_df.to_csv(OUTPUT_DATA_PATH, index=False)
    print(f"\nSuccess: Final QA-filtered dataset exported to '{OUTPUT_DATA_PATH}'.")
    print("The Spec-Driven Development pipeline is now completely executed.")
except Exception as e:
    print(f"\nError exporting the final dataset: {e}")